## 00 — Bounding Boxes

We have four simplified LOD files. Each still contains thousands of features spread across the globe. When a user is looking at Western Europe at zoom 8, there is no reason to send Siberian railroads to the renderer.

The first tool for eliminating invisible features is the **bounding box** — the smallest axis-aligned rectangle that fully contains a geometry.

This notebook covers:
1. What a bounding box is and how it is stored
2. How to compute one from a feature's coordinates
3. Why the railroad dataset already has them — and what to do with that

## What Is a Bounding Box?

An **axis-aligned bounding box (AABB)** is defined by four values:

```
[lon_min, lat_min, lon_max, lat_max]
```

This is also the GeoJSON `bbox` convention. Every GeoJSON object can optionally carry a `bbox` field with this exact format.

```
lat_max  ┌───────────────┐
         │               │
         │   feature     │
         │               │
lat_min  └───────────────┘
      lon_min          lon_max
```

The bounding box does not describe the shape of the feature — only its **extent**. Two very different shapes can have identical bounding boxes.

## The Railroad Dataset Already Has Bounding Boxes

Recall from Module 00 that each feature in `ne_10m_railroads.geojson` has a `bbox` key.

Let's inspect it.

In [9]:
import json
from pathlib import Path

data_path = Path("../../data/ne_10m_railroads.geojson")
with open(data_path) as f:
    railroads = json.load(f)

feature = railroads["features"][0]

print("Feature keys:", list(feature.keys()))
print("bbox:", feature["bbox"])
print()
print("Format: [lon_min, lat_min, lon_max, lat_max]")

Feature keys: ['type', 'properties', 'bbox', 'geometry']
bbox: [30.730275, 69.448054, 30.782502, 69.461111]

Format: [lon_min, lat_min, lon_max, lat_max]


The `bbox` field is precomputed and trustworthy for the raw data.

However, our LOD files were written by the pipeline in the previous module — without `bbox` fields. So we need to be able to **compute** a bounding box from coordinates ourselves.

## Computing a Bounding Box

Given a list of `[lon, lat]` coordinate pairs, the bounding box is simply the min and max of each axis.

In [10]:
def feature_bbox(feature):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for a GeoJSON LineString feature.
    """
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

In [11]:
# Verify our result matches the precomputed bbox
computed  = feature_bbox(feature)
precomputed = feature["bbox"]

print("Computed:    ", computed)
print("Precomputed: ", precomputed)
print("Match:", computed == precomputed)

Computed:     [30.730275, 69.448054, 30.782502, 69.461111]
Precomputed:  [30.730275, 69.448054, 30.782502, 69.461111]
Match: True


## Visualizing a Feature and Its Bounding Box

Let's display one feature and its bounding box on a map to see what it looks like.

In [12]:
from ipyleaflet import Map, GeoJSON

# Pick a longer feature for a more interesting bbox
long_features = sorted(railroads["features"], key=lambda f: len(f["geometry"]["coordinates"]), reverse=True)
f = long_features[2]

bbox = feature_bbox(f)
lon_min, lat_min, lon_max, lat_max = bbox

# Build the bbox as a GeoJSON polygon
bbox_polygon = {
    "type": "Feature",
    "properties": {"name": "bounding box"},
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min],
        ]]
    }
}

center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = Map(center=[center_lat, center_lon], zoom=5)

m.add(GeoJSON(data={"type": "FeatureCollection", "features": [f]},
              style={"color": "#cc3300", "weight": 2}))
m.add(GeoJSON(data={"type": "FeatureCollection", "features": [bbox_polygon]},
              style={"color": "#0066cc", "weight": 1.5, "fillOpacity": 0.05}))
m

Map(center=[63.0397215, 75.576944], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

## Bounding Boxes for the LOD Files

Our LOD output files do not have precomputed `bbox` fields. We will compute them on the fly during culling.

As an optimization preview: we could precompute and store bounding boxes once at pipeline time, then just read the stored values during culling. This is a common real-world pattern.

For now, let's verify the function works on a LOD feature.

In [13]:
lod_path = Path("../../data/lod/railroads_fine.geojson")
with open(lod_path) as f:
    fine = json.load(f)

sample = fine["features"][100]
bbox = feature_bbox(sample)

print("LOD feature bbox:", bbox)
print("Coordinate count:", len(sample["geometry"]["coordinates"]))

LOD feature bbox: [66.311406, 66.714926, 68.935817, 68.190447]
Coordinate count: 26


## Exercise A

Write a function `collection_bbox(features)` that returns the bounding box of an **entire FeatureCollection** — the smallest rectangle that contains all features.

Apply it to each of the four LOD files and compare the results. Do they all cover the same geographic extent?

In [17]:
def collection_bbox(features):
    """
    Compute [lon_min, lat_min, lon_max, lat_max] for an entire FeatureCollection.
    """
    all_lons = []
    all_lats = []

    for feature in features:
        coords = feature["geometry"]["coordinates"]

        for lon, lat in coords:
            all_lons.append(lon)
            all_lats.append(lat)

    return [min(all_lons), min(all_lats), max(all_lons), max(all_lats)]


# Load all LOD files
lod_data = {}
for level in ["coarse", "medium", "fine", "extra_fine"]:
    lod_file = Path(f"../../data/lod/railroads_{level}.geojson")
    with open(lod_file) as f:
        lod_data[level] = json.load(f)

print(f"{'Level':<12} {'Bounding Box [lon_min, lat_min, lon_max, lat_max]'}")
print("-" * 80)

bboxes = {}

for level in ["coarse", "medium", "fine", "extra_fine"]:
    bbox = collection_bbox(lod_data[level]["features"])
    bboxes[level] = bbox

    print(f"{level:<12} {bbox}")


print("\nDo all LOD levels have the same bbox?")
print(len(set(tuple(bbox) for bbox in bboxes.values())) == 1)

Level        Bounding Box [lon_min, lat_min, lon_max, lat_max]
--------------------------------------------------------------------------------
coarse       [-123.014722, -41.475186, 150.961667, 60.976516]
medium       [-150.112222, -51.894722, 179.357778, 69.604375]
fine         [-150.112222, -51.894722, 179.357778, 69.604375]
extra_fine   [-150.112222, -51.895278, 179.357778, 69.604375]

Do all LOD levels have the same bbox?
False


They may not all cover the exact same geographic extent because the coarse level uses a `scalerank` filter and drops many less-important features. The medium, fine, and extra fine levels should be closer to the original extent because they keep all features and mainly change the coordinate detail.

## Exercise B

Find the **5 features with the largest bounding box area** in the fine LOD file.

Bounding box area = `(lon_max - lon_min) * (lat_max - lat_min)`.

Print each one's bbox area and its `category` property. Do the results make geographic sense?

In [18]:
def bbox_area(bbox):
    lon_min, lat_min, lon_max, lat_max = bbox
    return (lon_max - lon_min) * (lat_max - lat_min)


feature_areas = []

for feature in lod_data["fine"]["features"]:
    bbox = feature_bbox(feature)
    area = bbox_area(bbox)

    category = feature["properties"].get("category", "unknown")

    feature_areas.append((area, category, bbox))

# Sort largest to smallest
feature_areas.sort(reverse=True, key=lambda x: x[0])

print("Top 5 features with the largest bbox area:\n")

for i, (area, category, bbox) in enumerate(feature_areas[:5], start=1):
    print(f"{i}. Area: {area:.2f}")
    print(f"   Category: {category}")
    print(f"   BBox: {bbox}\n")

Top 5 features with the largest bbox area:

1. Area: 29.31
   Category: 0
   BBox: [90.609351, 29.645107, 94.942815, 36.409271]

2. Area: 18.65
   Category: 0
   BBox: [132.255704, -23.549819, 134.323448, -14.530934]

3. Area: 17.70
   Category: 3
   BBox: [-64.094167, -26.192499, -58.156944, -23.210555]

4. Area: 17.39
   Category: 2
   BBox: [14.021914, 54.802728, 20.000001, 57.711109]

5. Area: 12.35
   Category: 2
   BBox: [-49.137778, -5.491666, -44.372222, -2.899167]



The results should make geographic sense because the largest bounding boxes usually belong to long railroad lines that span large regions or countries. Features that cross wide east-west or north-south distances naturally produce larger bbox areas.

## Check Your Understanding

Two different railroad features can have identical bounding boxes even though they follow completely different paths.

Describe a scenario where this happens — what would the two features look like? And does this cause any problem for our culling system?

---

Two railroad features could start and end in the same general region but take very different routes inside the box. For example, one line might curve along a coastline while another zigzags through mountains, yet both still fit inside the same minimum and maximum longitude/latitude values. This is not a major problem for the culling system because bounding boxes are only used as a fast first check to decide whether a feature might be visible on screen.

## Next

In [01 — Intersection Test](./01-Intersection_Test.ipynb), we write the function that checks whether a feature's bounding box overlaps the current viewport.